Menyiapkan Dataset

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


A. Feature Engineering 

Gunakan StringIndexer untuk mengubah kolom (nama_menu
indeks numerik. Gabungkan seluruh fitur ( qty , harga_satuan , dan ketiga kolom hasil indexing) menjadi
satu kolom features menggunakan VectorAssembler.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("TugasMandiriPertemuan11") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca dataset
df = spark.read.csv(
    "tugas11_penjualan.csv",
    header=True,
    inferSchema=True
)

# Menampilkan struktur dan beberapa data
df.printSchema()
df.show(5)

26/11/12 08:50:12 WARN Utils: Your hostname, asih2 resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/12 08:50:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 08:50:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



In [3]:
# StringIndexer untuk kolom kategorikal
indexer_nama_menu = StringIndexer(
    inputCol="nama_menu",
    outputCol="nama_menu_idx"
)

indexer_cabang = StringIndexer(
    inputCol="cabang",
    outputCol="cabang_idx"
)

indexer_waktu = StringIndexer(
    inputCol="waktu_pesan",
    outputCol="waktu_pesan_idx"
)

# Fit dan transform menggunakan model StringIndexer
indexer_nama_menu_model = indexer_nama_menu.fit(df)
df_indexed = indexer_nama_menu_model.transform(df)

indexer_cabang_model = indexer_cabang.fit(df_indexed)
df_indexed = indexer_cabang_model.transform(df_indexed)

indexer_waktu_model = indexer_waktu.fit(df_indexed)
df_indexed = indexer_waktu_model.transform(df_indexed)

# VectorAssembler
assembler = VectorAssembler(
    inputCols=[
        "qty",
        "harga_satuan",
        "nama_menu_idx",
        "cabang_idx",
        "waktu_pesan_idx"
    ],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)

# Menampilkan hasil feature engineering
df_final.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "nama_menu_idx",
    "cabang_idx",
    "waktu_pesan_idx",
    "features",
    "total_penjualan"
).show(10, truncate=False)

+--------------+----------+-----------+---+------------+-------------+----------+---------------+-------------------------+------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|nama_menu_idx|cabang_idx|waktu_pesan_idx|features                 |total_penjualan   |
+--------------+----------+-----------+---+------------+-------------+----------+---------------+-------------------------+------------------+
|Kentang Goreng|Semarang  |Malam      |1  |30000       |5.0          |2.0       |1.0            |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|Kentang Goreng|Yogyakarta|Sore       |3  |20000       |5.0          |1.0       |3.0            |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|Kentang Goreng|Semarang  |Siang      |5  |22000       |5.0          |2.0       |2.0            |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|Kentang Goreng|Yogyakarta|Pagi       |2  |20000       |5.0          |1.0       |0.0            |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|

StringIndexer digunakan untuk mengubah data kategorikal berupa nama menu, cabang, dan waktu pemesanan menjadi nilai numerik. Hal ini diperlukan karena model Machine Learning tidak dapat menggunakan data berupa teks secara langsung.  Setelah itu, VectorAssembler menggabungkan qty, harga_satuan, nama_menu_idx, cabang_idx, dan waktu_pesan_idx menjadi satu kolom features. Kolom tersebut menjadi input yang digunakan oleh model LinearRegression.

Berdasarkan output printSchema(), dataset berhasil dibaca dengan jumlah enam kolom, yaitu nama_menu, cabang, waktu_pesan, qty, harga_satuan, dan total_penjualan.

Tipe data sudah sesuai. Kolom kategorikal seperti nama_menu, cabang, dan waktu_pesan memiliki tipe string, sedangkan qty dan harga_satuan bertipe integer. Kolom total_penjualan bertipe double karena memiliki nilai desimal.

Setelah dilakukan proses StringIndexer, ketiga kolom kategorikal berhasil diubah menjadi angka indeks. Contohnya, Kentang Goreng memiliki indeks 5, Semarang memiliki indeks 2, dan Malam memiliki indeks 1. Nilai indeks tersebut ditentukan berdasarkan frekuensi kemunculan kategori dalam dataset.

Selanjutnya, VectorAssembler berhasil menggabungkan lima fitur menjadi satu kolom features. Contohnya, data pertama menghasilkan vector [1.0,30000.0,5.0,2.0,1.0].

Dengan demikian, proses feature engineering berhasil dilakukan dan data sudah siap digunakan untuk pelatihan model Machine Learning.

B. Train/Test Split & Pelatihan Model (bobot 20%)

Bagi data menjadi 75% training dan 25% testing ( seed=101 ). Latih model LinearRegression dengan
labelCol="total_penjualan"

In [5]:
from pyspark.ml.regression import LinearRegression

# Membagi data menjadi 75% training dan 25% testing
train, test = df_final.randomSplit(
    [0.75, 0.25],
    seed=101
)

print("Jumlah data training :", train.count())
print("Jumlah data testing  :", test.count())

# Membuat model Linear Regression
lr = LinearRegression(
    featuresCol="features",
    labelCol="total_penjualan"
)

# Melatih model
model = lr.fit(train)

print("\nModel berhasil dilatih.")
print("Koefisien model :", model.coefficients)
print("Intercept       :", model.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661



Model berhasil dilatih.
Koefisien model : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept       : -66903.17766485889


Data dibagi menjadi dua bagian, yaitu 75% untuk training dan 25% untuk testing. Data training digunakan oleh model untuk mempelajari hubungan antara fitur dengan total_penjualan.

Parameter seed=101 digunakan agar pembagian data tetap konsisten ketika notebook dijalankan kembali.

Model yang digunakan adalah LinearRegression. Kolom features menjadi input model, sedangkan total_penjualan menjadi nilai yang ingin diprediksi.

Berdasarkan hasil pembagian dataset, diperoleh sebanyak 1.839 baris data training dan 661 baris data testing.

Pembagian tersebut sudah mendekati proporsi 75% untuk training dan 25% untuk testing sesuai instruksi soal. Perbedaan jumlah baris terjadi karena proses randomSplit() membagi data secara acak.

Model LinearRegression berhasil dilatih menggunakan lima fitur yang telah digabungkan ke dalam kolom features, dengan total_penjualan sebagai target prediksi.

Hasil pelatihan menghasilkan koefisien [22046.66, 3.02, -123.21, 279.28, 339.64] dan intercept sebesar -66903.18. Koefisien menunjukkan arah dan besarnya hubungan masing-masing fitur terhadap hasil prediksi model.

Dengan demikian, proses pembagian data dan pelatihan model berhasil dilakukan tanpa error.

C. Evaluasi Model (bobot 20%)

Hitung dan tampilkan RMSE dan R2 dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah
model ini cukup baik untuk dipakai memprediksi omzet?


In [6]:
from pyspark.ml.evaluation import RegressionEvaluator

# Membuat prediksi menggunakan data testing
predictions = model.transform(test)

# Evaluator RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator_rmse.evaluate(predictions)

# Evaluator R²
evaluator_r2 = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

# Menampilkan beberapa hasil prediksi
predictions.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "total_penjualan",
    "prediction"
).show(10, truncate=False)

RMSE : Rp 10,900.80
R²   : 0.9107
+---------+--------+-----------+---+------------+------------------+------------------+
|nama_menu|cabang  |waktu_pesan|qty|harga_satuan|total_penjualan   |prediction        |
+---------+--------+-----------+---+------------+------------------+------------------+
|Americano|Magelang|Malam      |1  |25000       |19721.170446051627|31056.146105506836|
|Americano|Magelang|Malam      |1  |25000       |21236.0956379149  |31056.146105506836|
|Americano|Magelang|Malam      |4  |22000       |85834.8564401191  |88127.37331410211 |
|Americano|Magelang|Malam      |4  |22000       |102174.09100317833|88127.37331410211 |
|Americano|Magelang|Malam      |4  |30000       |113044.63431555482|112310.74025938749|
|Americano|Magelang|Malam      |5  |22000       |115810.29600214415|110174.03658512786|
|Americano|Magelang|Malam      |5  |25000       |140386.40273615732|119242.79918960987|
|Americano|Magelang|Pagi       |2  |30000       |73842.64420790318 |67877.77492201277 

RMSE menunjukkan rata-rata besar kesalahan prediksi model dalam satuan total_penjualan. Semakin kecil nilai RMSE, semakin dekat hasil prediksi model dengan nilai sebenarnya.

R² menunjukkan seberapa besar variasi total_penjualan yang dapat dijelaskan oleh model. Nilai R² yang semakin mendekati 1 menunjukkan model yang semakin baik.

Berdasarkan hasil evaluasi pada data testing, diperoleh nilai:
- RMSE: Rp10.900,80
- R²: 0,9107
Nilai RMSE sebesar Rp10.900,80 menunjukkan bahwa rata-rata kesalahan prediksi model sekitar Rp10.900,80 dalam satuan penjualan. Semakin kecil nilai RMSE, semakin dekat hasil prediksi dengan nilai sebenarnya.

Sementara itu, nilai R² sebesar 0,9107 menunjukkan bahwa model mampu menjelaskan sekitar 91,07% variasi total penjualan berdasarkan fitur yang digunakan. Sisanya sekitar 8,93% dipengaruhi oleh faktor lain yang belum dimasukkan ke dalam model atau variasi acak pada data.

Berdasarkan hasil tersebut, model dapat dikatakan memiliki performa yang cukup baik untuk memprediksi total penjualan karena nilai R² mendekati 1 dan tingkat kesalahan prediksinya relatif kecil dibandingkan skala penjualan pada dataset.

Contohnya, pada salah satu data pengujian, nilai penjualan sebenarnya sebesar Rp19.721,17, sedangkan hasil prediksi model sebesar Rp31.056,15. Hal ini menunjukkan bahwa meskipun model memiliki performa keseluruhan yang baik, masih terdapat beberapa prediksi yang memiliki selisih cukup besar.

D. Eksperimen Perbandingan Fitur (bobot 25%)

Latih model kedua yang HANYA menggunakan fitur qty dan harga_satuan (tanpa informasi
menu/cabang/waktu). Bandingkan R2 model kedua ini dengan model utama pada bagian C. Tulis
kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi
prediksi secara berarti?

In [8]:
# Membuat VectorAssembler untuk model kedua
assembler_sederhana = VectorAssembler(
    inputCols=[
        "qty",
        "harga_satuan"
    ],
    outputCol="features_sederhana"
)

df_sederhana = assembler_sederhana.transform(df_final)

# Membagi data dengan proporsi dan seed yang sama
train_sederhana, test_sederhana = df_sederhana.randomSplit(
    [0.75, 0.25],
    seed=101
)

# Membuat Linear Regression kedua
lr_sederhana = LinearRegression(
    featuresCol="features_sederhana",
    labelCol="total_penjualan"
)

# Melatih model kedua
model_sederhana = lr_sederhana.fit(train_sederhana)

# Prediksi
predictions_sederhana = model_sederhana.transform(test_sederhana)

# Menghitung R²
evaluator_r2_sederhana = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

r2_sederhana = evaluator_r2_sederhana.evaluate(
    predictions_sederhana
)

# Menghitung RMSE model kedua
evaluator_rmse_sederhana = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="rmse"
)

rmse_sederhana = evaluator_rmse_sederhana.evaluate(
    predictions_sederhana
)

print(f"R² model utama     : {r2:.4f}")
print(f"R² model sederhana : {r2_sederhana:.4f}")

print(f"\nRMSE model utama     : Rp {rmse:,.2f}")
print(f"RMSE model sederhana : Rp {rmse_sederhana:,.2f}")

selisih_r2 = r2 - r2_sederhana

print(f"\nSelisih R² : {selisih_r2:.4f}")

if r2 > r2_sederhana:
    print(
        "Kesimpulan: model utama memiliki R² lebih tinggi "
        "daripada model sederhana."
    )
elif r2 < r2_sederhana:
    print(
        "Kesimpulan: model sederhana memiliki R² lebih tinggi "
        "daripada model utama."
    )
else:
    print(
        "Kesimpulan: kedua model memiliki nilai R² yang sama."
    )

R² model utama     : 0.9107
R² model sederhana : 0.9108

RMSE model utama     : Rp 10,900.80
RMSE model sederhana : Rp 10,896.03

Selisih R² : -0.0001
Kesimpulan: model sederhana memiliki R² lebih tinggi daripada model utama.


Pada model kedua hanya digunakan dua fitur, yaitu qty dan harga_satuan. Informasi mengenai nama menu, cabang, dan waktu pemesanan tidak digunakan.

Nilai R² kedua model kemudian dibandingkan. Jika R² model utama lebih tinggi, berarti penambahan informasi menu, cabang, dan waktu memberikan informasi tambahan terhadap prediksi omzet. Jika selisihnya kecil, berarti fitur tambahan tersebut tidak memberikan peningkatan akurasi yang terlalu berarti.

Berdasarkan hasil perbandingan kedua model, diperoleh:

Model utama (5 fitur):
- R² = 0,9107
- RMSE = Rp10.900,80
Model sederhana (2 fitur):
- R² = 0,9108
- RMSE = Rp10.896,03
Selisih nilai R² kedua model adalah -0,0001. Hasil tersebut menunjukkan bahwa model sederhana memiliki nilai R² sedikit lebih tinggi dibandingkan model utama.

Namun, perbedaan performa kedua model sangat kecil dan hampir tidak signifikan. Hal ini menunjukkan bahwa penambahan informasi nama_menu, cabang, dan waktu_pesan belum memberikan peningkatan akurasi prediksi yang berarti.

Kondisi tersebut dapat terjadi karena nilai total_penjualan pada dataset sebagian besar dibentuk dari perkalian qty dengan harga_satuan, ditambah sedikit variasi acak. Oleh karena itu, kedua fitur tersebut sudah cukup untuk menjelaskan sebagian besar perubahan nilai penjualan.

Kesimpulan: Model sederhana yang hanya menggunakan qty dan harga_satuan memiliki performa sedikit lebih baik dibandingkan model utama. Dengan demikian, penambahan fitur menu, cabang, dan waktu pemesanan belum memberikan peningkatan akurasi yang berarti pada dataset ini.

E. Prediksi Skenario Baru (bobot 15%)
Buat sebuah DataFrame baru berisi 3 skenario pesanan (data dummy) (boleh bebas kombinasi
menu/cabang/waktu/qty/harga, namun harus melalui tahapan StringIndexer dan VectorAssembler
yang sama seperti data training - perhatikan: gunakan model StringIndexer yang sama yang sudah di-
fit() sebelumnya, jangan membuat StringIndexer baru). Tampilkan hasil prediksi total_penjualan
untuk ketiga skenario tersebut.

In [9]:
# Membuat 3 skenario pesanan baru
data_skenario = [
    ("Kopi Susu", "Magelang", "Pagi", 2, 20000),
    ("Cappuccino", "Yogyakarta", "Sore", 3, 25000),
    ("Roti Bakar", "Semarang", "Malam", 1, 22000)
]

kolom_skenario = [
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan"
]

df_skenario = spark.createDataFrame(
    data_skenario,
    kolom_skenario
)

print("Data skenario baru:")
df_skenario.show()

Data skenario baru:


+----------+----------+-----------+---+------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|
+----------+----------+-----------+---+------------+
| Kopi Susu|  Magelang|       Pagi|  2|       20000|
|Cappuccino|Yogyakarta|       Sore|  3|       25000|
|Roti Bakar|  Semarang|      Malam|  1|       22000|
+----------+----------+-----------+---+------------+



In [10]:
# Menggunakan model StringIndexer yang SAMA
# dengan yang sudah di-fit pada Soal 1

df_skenario_indexed = indexer_nama_menu_model.transform(
    df_skenario
)

df_skenario_indexed = indexer_cabang_model.transform(
    df_skenario_indexed
)

df_skenario_indexed = indexer_waktu_model.transform(
    df_skenario_indexed
)

# Menggunakan VectorAssembler yang SAMA
df_skenario_final = assembler.transform(
    df_skenario_indexed
)

# Melakukan prediksi menggunakan model utama
prediksi_skenario = model.transform(
    df_skenario_final
)

# Menampilkan hasil prediksi
prediksi_skenario.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "prediction"
).show(
    3,
    truncate=False
)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |20000       |37525.35198441948 |
|Cappuccino|Yogyakarta|Sore       |3  |25000       |75861.60625084834 |
|Roti Bakar|Semarang  |Malam      |1  |22000       |22053.095526312085|
+----------+----------+-----------+---+------------+------------------+



iga data pesanan baru dibuat sebagai skenario prediksi. Data tersebut tidak mempunyai nilai total_penjualan karena nilai tersebut justru ingin diprediksi oleh model.

Sesuai ketentuan soal, digunakan model StringIndexer yang sama dengan yang telah dibuat pada proses training. Hal ini penting agar pemetaan kategori antara data training dan data baru tetap konsisten.

Setelah data kategorikal diubah menjadi indeks, VectorAssembler yang sama digunakan untuk membentuk kolom features. Selanjutnya model LinearRegression utama menghasilkan nilai prediction sebagai estimasi total_penjualan.

Berdasarkan hasil pengujian, tiga skenario pesanan berhasil diproses dan menghasilkan prediksi menggunakan model utama.

Skenario pertama adalah pesanan Kopi Susu dari cabang Magelang pada pagi hari dengan jumlah 2 unit dan harga satuan Rp20.000. Model menghasilkan prediksi total penjualan sebesar Rp37.525,35.

Skenario kedua adalah pesanan Cappuccino dari cabang Yogyakarta pada sore hari dengan jumlah 3 unit dan harga satuan Rp25.000. Model menghasilkan prediksi sebesar Rp75.861,61.

Skenario ketiga adalah pesanan Roti Bakar dari cabang Semarang pada malam hari dengan jumlah 1 unit dan harga satuan Rp22.000. Model menghasilkan prediksi sebesar Rp22.053,10.

Ketiga skenario berhasil diproses menggunakan model StringIndexer yang sama dengan proses training. Selain itu, VectorAssembler yang sama juga digunakan untuk menggabungkan fitur sebelum dilakukan prediksi.

Hasil tersebut menunjukkan bahwa model yang telah dilatih mampu menerima data pesanan baru dan menghasilkan estimasi total penjualan berdasarkan atribut yang diberikan.

In [11]:
print("===== KESIMPULAN =====")
print(f"R² model utama     : {r2:.4f}")
print(f"R² model sederhana : {r2_sederhana:.4f}")
print(f"RMSE model utama   : Rp {rmse:,.2f}")

if r2 > r2_sederhana:
    print(
        "Model utama lebih baik karena memiliki nilai R² "
        "yang lebih tinggi."
    )
elif r2 < r2_sederhana:
    print(
        "Model sederhana lebih baik karena memiliki nilai R² "
        "yang lebih tinggi."
    )
else:
    print(
        "Kedua model memiliki performa R² yang sama."
    )

===== KESIMPULAN =====
R² model utama     : 0.9107
R² model sederhana : 0.9108
RMSE model utama   : Rp 10,900.80
Model sederhana lebih baik karena memiliki nilai R² yang lebih tinggi.


Berdasarkan seluruh tahapan praktikum, proses pembuatan model prediksi menggunakan Spark MLlib berhasil dilakukan, mulai dari feature engineering, pembagian data, pelatihan model, evaluasi, perbandingan fitur, hingga prediksi skenario baru.

Model utama menghasilkan nilai RMSE sebesar Rp10.900,80 dan R² sebesar 0,9107. Hasil ini menunjukkan bahwa model mampu menjelaskan sekitar 91,07% variasi total penjualan sehingga memiliki performa prediksi yang cukup baik.

Pada eksperimen perbandingan fitur, model sederhana yang hanya menggunakan qty dan harga_satuan memperoleh R² sebesar 0,9108, sedikit lebih tinggi dibandingkan model utama. Perbedaan tersebut sangat kecil sehingga penambahan informasi menu, cabang, dan waktu pemesanan belum memberikan peningkatan akurasi yang berarti.

Secara keseluruhan, praktikum ini menunjukkan bahwa Spark MLlib dapat digunakan untuk membangun model regresi dalam memprediksi penjualan. Hasil eksperimen juga menunjukkan bahwa pemilihan fitur yang tepat dapat menghasilkan model yang lebih sederhana tanpa mengurangi kemampuan prediksinya secara berarti.

In [12]:
# Menutup SparkSession setelah seluruh tugas selesai
spark.stop()

print("SparkSession ditutup.")

SparkSession ditutup.
